In [24]:
import statsmodels, numpy, pandas, scipy
print("statsmodels:",statsmodels.__version__)
print("ready")

statsmodels: 0.14.6
ready


In [25]:
## Tools for power analysis
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

## Current checkout conversion rate 
baseline_rate=0.10

## Catching B converting at 12% vs A at 10% 
mde= 0.02

## False Positive tolerance
alpha= 0.05

##Chances of catching a real effect when one exists
power= 0.80

## Converting two rates into single effect size number
effect_size = proportion_effectsize(baseline_rate + mde, baseline_rate)

## Calculating required sample size
analysis = NormalIndPower()
n_per_group = analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)

print(f"Effect size: {effect_size:.4f}")
print(f"Required users per group: {n_per_group:.0f}")
print(f"Total users needed: {2 * n_per_group:.0f}")

Effect size: 0.0640
Required users per group: 3835
Total users needed: 7669


In [11]:
## Observing N when moving each lever
scenarios = [
    # (label, baseline, mde, alpha, power)
    ("Baseline",              0.10, 0.02, 0.05, 0.80),
    ("Smaller MDE (1%)",      0.10, 0.01, 0.05, 0.80),
    ("Larger MDE (5%)",       0.10, 0.05, 0.05, 0.80),
    ("Higher power (90%)",    0.10, 0.02, 0.05, 0.90),
    ("Stricter alpha (1%)",   0.10, 0.02, 0.01, 0.80),
    ("Looser alpha (10%)",    0.10, 0.02, 0.10, 0.80),
]

print(f"{'Scenario':<25} {'N per group':>12} {'Total N':>10}")
print("-" * 50)

for label, base, mde, a, pwr in scenarios:
    es = proportion_effectsize(base + mde, base) #effect size
    n = NormalIndPower().solve_power(
        effect_size=es, alpha=a, power=pwr, alternative='two-sided' #sample size calculation
    )
    print(f"{label:<25} {n:>12.0f} {2*n:>10.0f}")

Scenario                   N per group    Total N
--------------------------------------------------
Baseline                          3835       7669
Smaller MDE (1%)                 14744      29488
Larger MDE (5%)                    680       1361
Higher power (90%)                5133      10267
Stricter alpha (1%)               5706      11412
Looser alpha (10%)                3020       6041


## Step 1: Power Analysis

Before collecting any data, we determine how many users we need.

**Business context:** We're testing a new checkout page design.  
- Baseline conversion rate: 10%  
- Minimum Detectable Effect: 2 percentage points (business decision)  
- Alpha: 0.05 (5% false positive tolerance)  
- Power: 0.80 (80% chance of catching a real effect)  

**Result: 3,835 users per group (7,669 total)**

Key insight: MDE is the dominant lever. Halving the MDE (1%) nearly 
quadruples the required sample size. This is a business trade-off, 
not a statistical one — smaller effects require more data to distinguish 
from noise.

We commit to this sample size before looking at any results. 
Peeking early inflates false positive rate above our 5% threshold.

In [26]:
import numpy as np
from scipy import stats 

## Setting a random seed 
np.random.seed(42)

true_rate_A = 0.10
true_rate_B = 0.12
n= 3835

group_A = np.random.binomial(n=1, p=true_rate_A, size=n)
group_B = np.random.binomial(n=1, p=true_rate_B, size=n)

## Observations
conversions_A = group_A.sum()
conversions_B = group_B.sum()
observed_rate_A = conversions_A / n
observed_rate_B = conversions_B / n

print("=== OBSERVED DATA (what the analyst sees) ===")
print(f"Group A: {conversions_A} conversions / {n} visitors = {observed_rate_A: .3f}")
print(f"Group B: {conversions_B} conversions / {n} visitors = {observed_rate_B: .3f}")
print(f"Observed lift: {observed_rate_B - observed_rate_A: .3f}")

=== OBSERVED DATA (what the analyst sees) ===
Group A: 368 conversions / 3835 visitors =  0.096
Group B: 450 conversions / 3835 visitors =  0.117
Observed lift:  0.021


In [6]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

## Two proportion Z test
conversions = np.array([conversions_A, conversions_B])
totals = np.array([n,n])

stat, p_value = proportions_ztest(conversions, totals, alternative='two-sided')

## Confidence interval on the lift
from statsmodels.stats.proportion import proportion_confint

ci_A = proportion_confint(conversions_A, n, alpha =0.05, method= 'normal')
ci_B = proportion_confint(conversions_B, n, alpha =0.05, method= 'normal')

lift = observed_rate_B - observed_rate_A
lift_ci_low = ci_B[0] - ci_A[1]
lift_ci_high = ci_B[1] - ci_A[0]

## Decision
print("=== SIGNIFICANCE TEST ===")
print(f"Z-statistic:      {stat:.4f}")
print(f"P-value:          {p_value:.4f}")
print(f"Observed lift:    {lift:.4f}")
print(f"95% CI on lift:   [{lift_ci_low:.4f}, {lift_ci_high:.4f}")
print()

if p_value < 0.05:
    print("DECISION: Statistically significant. Evidence that B differs from A.")
    print ((f"We are 95% confident the true lift is between " f"{lift_ci_low:.3f} and {lift_ci_high:.3f}"))
else:
    print("DECISION: Not significant. Cannot distinguish B from A.")

=== SIGNIFICANCE TEST ===
Z-statistic:      -3.0334
P-value:          0.0024
Observed lift:    0.0214
95% CI on lift:   [0.0019, 0.0409

DECISION: Statistically significant. Evidence that B differs from A.
We are 95% confident the true lift is between 0.002 and 0.041


## Proof: What happens when you ignore the power analysis?

We rerun the exact same test but collect only half the users 
our power analysis recommended. This simulates peeking early 
or ignoring sample size requirements.

In [27]:
# Half the recommended sample size
np.random.seed(42)
n_half = n // 2   # 1917 users per group instead of 3835

group_A_small = np.random.binomial(n=1, p=true_rate_A, size=n_half)
group_B_small = np.random.binomial(n=1, p=true_rate_B, size=n_half)

conv_A_small = group_A_small.sum()
conv_B_small = group_B_small.sum()

stat_small, p_small = proportions_ztest(
    np.array([conv_A_small, conv_B_small]),
    np.array([n_half, n_half]),
    alternative='two-sided'
)

lift_small = (conv_B_small / n_half) - (conv_A_small / n_half)

print("=== UNDERPOWERED TEST (half the required sample) ===")
print(f"Users per group:  {n_half}")
print(f"Observed lift:    {lift_small:.4f}")
print(f"P-value:          {p_small:.4f}")
print()

if p_small < 0.05:
    print("DECISION: Significant — correctly detected the effect.")
else:
    print("DECISION: Not significant — missed a real 2% lift.")
    print("This is a FALSE NEGATIVE (Type II error).")
    print("B is genuinely better, but we lacked the power to see it.")

=== UNDERPOWERED TEST (half the required sample) ===
Users per group:  1917
Observed lift:    0.0110
P-value:          0.2709

DECISION: Not significant — missed a real 2% lift.
This is a FALSE NEGATIVE (Type II error).
B is genuinely better, but we lacked the power to see it.


## Key Finding: The Cost of Ignoring Power Analysis

With the correct sample (3,835/group):
- P-value: 0.0024 → Correctly detected the 2% lift
- Observed lift: 2.1%

With half the sample (1,917/group):
- P-value: 0.2709 → Missed the exact same 2% lift  
- Observed lift: 1.1% (noise distorted the estimate downward)

**This is a Type II error (false negative):** concluding there is no 
effect when one genuinely exists. The consequence in a real business: 
you'd fail to ship a genuinely better checkout page, leaving real 
revenue on the table.

The power analysis exists precisely to prevent this.

In [28]:
np.random.seed(123)

n_experiments = 1000
significant_count = 0
p_values = []

for i in range(n_experiments):
    # Simulate one experiment
    a = np.random.binomial(n=1, p=true_rate_A, size=n)
    b = np.random.binomial(n=1, p=true_rate_B, size=n)

    # Run the test
    _, p = proportions_ztest(
        np.array([a.sum(), b.sum()]),
        np.array([n, n]),
        alternative='two-sided'
    )

    # These THREE lines must be inside the loop
    p_values.append(p)
    if p < 0.05:
        significant_count += 1

# These lines are OUTSIDE the loop (no indentation)
empirical_power = significant_count / n_experiments

print(f"Experiments run:         {n_experiments}")
print(f"Significant results:     {significant_count}")
print(f"Empirical power:         {empirical_power:.3f}")
print(f"Theoretical power:       0.800")
print(f"Difference:              {abs(empirical_power - 0.80):.3f}")
print()
if 0.75 <= empirical_power <= 0.85:
    print("VERIFIED: Empirical power matches theoretical power.")
    print("The power analysis guarantee holds.")
else:
    print("Note: Empirical power outside expected range.")

Experiments run:         1000
Significant results:     777
Empirical power:         0.777
Theoretical power:       0.800
Difference:              0.023

VERIFIED: Empirical power matches theoretical power.
The power analysis guarantee holds.


## Step 2: Empirical Power Verification

We ran 1,000 independent simulated experiments, each with the same 
true effect (A=10%, B=12%) and the same sample size (3,835/group).

**Results:**
- Significant detections: 777 / 1,000
- Empirical power: 0.777
- Theoretical power: 0.800
- Difference: 0.023 (sampling variability, not error)

**Interpretation:** The power analysis guarantee holds. Across 1,000 
parallel universes with identical underlying truth, our test correctly 
detected the 2% lift ~80% of the time — exactly as the math predicted.

The 223 experiments that missed the effect were not errors in method — 
they were the expected 20% false negatives that power=0.80 explicitly 
accepts as a trade-off against sample size.

## Step 3: The Peeking Problem

What happens if we check results repeatedly before collecting 
the full sample? Even with no real effect, repeated peeking 
inflates our false positive rate well above the 5% we intended.

In [29]:
np.random.seed(42)

## No real effect this time as both groups are identical
true_rate_null = 0.10
n_peek_experiments = 1000

## Checkpoints: peek at 20%, 40%, 60%, 80%, 100% of data
checkpoints = [0.2, 0.4, 0.6, 0.8, 1.0]
## How many users collected at each checkpoint
checkpoint_ns = [int(n * c) for c in checkpoints]

## For each experiment, did we declare significance at ANY checkpoint?
peeked_significant = 0
## For comparison: only looking at the end (correct approach)
end_only_significant = 0

for i in range(n_peek_experiments):
    ## Generating the FULL dataset upfront for both groups
    a_full = np.random.binomial(n=1, p=true_rate_null, size=n)
    b_full = np.random.binomial(n=1, p=true_rate_null, size=n)

    declared_significant = False

    for checkpoint_n in checkpoint_ns:
        # Look at only the first checkpoint_n users
        a_peek = a_full[:checkpoint_n]
        b_peek = b_full[:checkpoint_n]

        _, p_peek = proportions_ztest(
            np.array([a_peek.sum(), b_peek.sum()]),
            np.array([checkpoint_n, checkpoint_n]),
            alternative='two-sided'
        )

        # Peeking rule: stop and declare winner at first significant result
        if p_peek < 0.05 and not declared_significant:
            declared_significant = True

    if declared_significant:
        peeked_significant += 1

    ## Correct approach: only check at the end
    _, p_end = proportions_ztest(
        np.array([a_full.sum(), b_full.sum()]),
        np.array([n, n]),
        alternative='two-sided'
    )
    if p_end < 0.05:
        end_only_significant += 1

peeked_fpr = peeked_significant / n_peek_experiments
end_only_fpr = end_only_significant / n_peek_experiments

print("=== PEEKING PROBLEM DEMONSTRATION ===")
print(f"True effect: NONE (both groups convert at {true_rate_null})")
print(f"Experiments: {n_peek_experiments}")
print()
print(f"False positive rate WITH peeking:    {peeked_fpr:.3f} ({peeked_significant}/1000)")
print(f"False positive rate WITHOUT peeking: {end_only_fpr:.3f} ({end_only_significant}/1000)")
print()
print(f"Peeking inflated false positives by: "
      f"{(peeked_fpr - end_only_fpr):.3f} percentage points")
print()
if peeked_fpr > end_only_fpr + 0.05:
    print("CONFIRMED: Peeking meaningfully inflates false positive rate.")
    print("You would ship changes that do nothing — repeatedly.")

=== PEEKING PROBLEM DEMONSTRATION ===
True effect: NONE (both groups convert at 0.1)
Experiments: 1000

False positive rate WITH peeking:    0.147 (147/1000)
False positive rate WITHOUT peeking: 0.051 (51/1000)

Peeking inflated false positives by: 0.096 percentage points

CONFIRMED: Peeking meaningfully inflates false positive rate.
You would ship changes that do nothing — repeatedly.


## Step 3 Finding: Peeking Triples Your False Positive Rate

With no real effect (A = B = 10% conversion):

| Approach | False Positives | False Positive Rate |
|---|---|---|
| Check only at end (correct) | 51 / 1000 | 5.1% |
| Peek at 5 checkpoints | 147 / 1000 | 14.7% |

**Peeking inflated false positives by 9.6 percentage points.**

Why: each peek is an independent chance to cross p < 0.05 by luck. 
Five peeks means five chances to get a random extreme result. The 
probabilities compound — your actual false positive rate becomes 
nearly 3x what you intended.

**The fix:** commit to your sample size upfront via power analysis, 
then look exactly once at the end. This is not a convention — it is 
what makes your alpha=0.05 claim mathematically honest.

---
## Phase 2: Real Data — E-commerce A/B Test

In Phase 1 we validated our pipeline against simulated data with a 
known ground truth. Now we apply the exact same pipeline to a real 
dataset where the outcome is unknown.

**Business context:** An e-commerce company tested a new landing page 
against the old one. Did the new page improve conversion?

**Our approach:** Run power analysis on the observed baseline BEFORE 
looking at outcomes. Then analyse. Then decide.

In [30]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/ab_data.csv')

# First look
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())
print("\nGroup counts:")
print(df['group'].value_counts())
print("\nLanding page counts:")
print(df['landing_page'].value_counts())
print("\nConverted value counts:")
print(df['converted'].value_counts())

Shape: (294478, 5)

Columns: ['user_id', 'timestamp', 'group', 'landing_page', 'converted']

First 5 rows:


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1



Group counts:
group
treatment    147276
control      147202
Name: count, dtype: int64

Landing page counts:
landing_page
old_page    147239
new_page    147239
Name: count, dtype: int64

Converted value counts:
converted
0    259241
1     35237
Name: count, dtype: int64


In [31]:
# Check for mismatched assignments
# In a clean experiment: control = old_page, treatment = new_page
mismatch = df[
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'control')   & (df['landing_page'] == 'new_page'))
]

print(f"Total rows:      {len(df)}")
print(f"Mismatched rows: {len(mismatch)}")
print(f"Mismatch rate:   {len(mismatch)/len(df)*100:.2f}%")
print()
print("Mismatch breakdown:")
print(mismatch.groupby(['group','landing_page']).size())

# Check for duplicate user IDs
dupes = df[df.duplicated('user_id', keep=False)]
print(f"\nDuplicate user IDs: {df['user_id'].nunique()} unique out of {len(df)} rows")
print(f"Users appearing more than once: {len(df) - df['user_id'].nunique()}")

Total rows:      294478
Mismatched rows: 3893
Mismatch rate:   1.32%

Mismatch breakdown:
group      landing_page
control    new_page        1928
treatment  old_page        1965
dtype: int64

Duplicate user IDs: 290584 unique out of 294478 rows
Users appearing more than once: 3894


In [32]:
# Step 1: Remove mismatched assignments
df_clean = df[
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'control')   & (df['landing_page'] == 'old_page'))
].copy()

# Step 2: Remove duplicate user IDs — keep first occurrence
df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')

# Step 3: Verify
print(f"Original rows:   {len(df)}")
print(f"After cleaning:  {len(df_clean)}")
print(f"Rows removed:    {len(df) - len(df_clean)}")
print()
print("Clean group counts:")
print(df_clean['group'].value_counts())
print()
print("Clean landing page counts:")
print(df_clean['landing_page'].value_counts())
print()

# Baseline conversion rates
control_rate   = df_clean[df_clean['group']=='control']['converted'].mean()
treatment_rate = df_clean[df_clean['group']=='treatment']['converted'].mean()

print(f"Control conversion rate:   {control_rate:.4f} ({control_rate*100:.2f}%)")
print(f"Treatment conversion rate: {treatment_rate:.4f} ({treatment_rate*100:.2f}%)")
print(f"Observed lift:             {(treatment_rate - control_rate)*100:.4f} pp")

Original rows:   294478
After cleaning:  290584
Rows removed:    3894

Clean group counts:
group
treatment    145310
control      145274
Name: count, dtype: int64

Clean landing page counts:
landing_page
new_page    145310
old_page    145274
Name: count, dtype: int64

Control conversion rate:   0.1204 (12.04%)
Treatment conversion rate: 0.1188 (11.88%)
Observed lift:             -0.1578 pp


In [33]:
## Power Analysis on Real Data
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Use the CONTROL rate as baseline (this is what we'd know before the experiment)
baseline_real   = control_rate      # 12.04%
mde_real        = 0.02              # same business assumption: 2pp lift worth detecting
alpha_real      = 0.05
power_real      = 0.80

effect_size_real = proportion_effectsize(baseline_real + mde_real, baseline_real)

n_required = NormalIndPower().solve_power(
    effect_size=effect_size_real,
    alpha=alpha_real,
    power=power_real,
    alternative='two-sided'
)

n_actual = len(df_clean[df_clean['group'] == 'control'])

print(f"Baseline conversion rate:  {baseline_real:.4f} ({baseline_real*100:.2f}%)")
print(f"MDE assumed:               {mde_real*100:.1f} percentage points")
print(f"Required N per group:      {n_required:.0f}")
print(f"Actual N per group:        {n_actual}")
print()

if n_actual >= n_required:
    surplus = n_actual - n_required
    print(f"STATUS: OVERPOWERED by {surplus:,} users per group")
    print(f"This experiment had MORE than enough users to detect a {mde_real*100:.0f}pp lift.")
    print(f"If a real 2pp lift existed, we would have detected it.")
else:
    deficit = n_required - n_actual
    print(f"STATUS: UNDERPOWERED by {deficit:,} users per group")
    print(f"This experiment lacked sufficient users to reliably detect a {mde_real*100:.0f}pp lift.")

Baseline conversion rate:  0.1204 (12.04%)
MDE assumed:               2.0 percentage points
Required N per group:      4444
Actual N per group:        145274

STATUS: OVERPOWERED by 140,830.30265174765 users per group
This experiment had MORE than enough users to detect a 2pp lift.
If a real 2pp lift existed, we would have detected it.


In [34]:
## Running the significance test
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import numpy as np

control_group   = df_clean[df_clean['group'] == 'control']
treatment_group = df_clean[df_clean['group'] == 'treatment']

conv_control   = control_group['converted'].sum()
conv_treatment = treatment_group['converted'].sum()
n_control      = len(control_group)
n_treatment    = len(treatment_group)

stat, p_value = proportions_ztest(
    np.array([conv_treatment, conv_control]),
    np.array([n_treatment, n_control]),
    alternative='two-sided'
)

ci_control   = proportion_confint(conv_control,   n_control,   alpha=0.05, method='normal')
ci_treatment = proportion_confint(conv_treatment, n_treatment, alpha=0.05, method='normal')

lift          = treatment_rate - control_rate
lift_ci_low   = ci_treatment[0] - ci_control[1]
lift_ci_high  = ci_treatment[1] - ci_control[0]

print("=== SIGNIFICANCE TEST — REAL DATA ===")
print(f"Control:   {conv_control:,} conversions / {n_control:,} users = {control_rate:.4f}")
print(f"Treatment: {conv_treatment:,} conversions / {n_treatment:,} users = {treatment_rate:.4f}")
print(f"Observed lift:    {lift*100:.4f} pp")
print(f"Z-statistic:      {stat:.4f}")
print(f"P-value:          {p_value:.4f}")
print(f"95% CI on lift:   [{lift_ci_low*100:.4f}, {lift_ci_high*100:.4f}] pp")
print()
if p_value < 0.05:
    print("STATISTICAL RESULT: Significant (p < 0.05)")
else:
    print("STATISTICAL RESULT: Not significant (p >= 0.05)")
print()
if abs(lift) >= 0.02:
    print("PRACTICAL RESULT: Lift meets MDE threshold — business relevant")
else:
    print("PRACTICAL RESULT: Lift below MDE threshold — NOT business relevant")
    print("Recommendation: Do NOT ship. Effect is statistically detectable")
    print("but practically meaningless. New page offers no meaningful improvement.")

=== SIGNIFICANCE TEST — REAL DATA ===
Control:   17,489 conversions / 145,274 users = 0.1204
Treatment: 17,264 conversions / 145,310 users = 0.1188
Observed lift:    -0.1578 pp
Z-statistic:      -1.3109
P-value:          0.1899
95% CI on lift:   [-0.4915, 0.1759] pp

STATISTICAL RESULT: Not significant (p >= 0.05)

PRACTICAL RESULT: Lift below MDE threshold — NOT business relevant
Recommendation: Do NOT ship. Effect is statistically detectable
but practically meaningless. New page offers no meaningful improvement.


## Phase 2 Findings: Real E-commerce A/B Test

### Data Quality
- 3,894 rows removed: mismatched group/page assignments + duplicate user IDs
- Clean dataset: 145,274 control / 145,310 treatment users

### Power Analysis (run before looking at outcomes)
- Baseline conversion rate: 12.04%
- Required N for 2pp MDE: 4,444 per group
- Actual N: 145,274 per group — **32x overpowered**
- Implication: if a 2pp lift existed, we would have detected it with certainty

### Significance Test Results
- Control: 12.04% | Treatment: 11.88%
- Observed lift: -0.16 pp
- P-value: 0.1899 — not significant
- 95% CI on lift: [-0.49pp, +0.18pp]

### Recommendation: Do Not Ship
The entire confidence interval sits below our pre-stated MDE of 2pp.
Even the most optimistic outcome (+0.18pp) is 11x below the threshold
we defined as business-relevant. The new page is functionally equivalent
to the old one.

**Key lesson:** With a 32x overpowered experiment, statistical 
significance becomes almost meaningless — the test can detect effects 
too small to matter. Always evaluate practical significance (does the 
effect exceed MDE?) alongside statistical significance (is p < 0.05?).
This is why you set MDE before the experiment, not after.